# MSCI EUR SMALL : composites factoriels et analyse incrémentale fondés sur les preuves par variable

Ce notebook construit uniquement un pipeline reproductible et n'est pas exécuté lors de sa création. Pour chaque famille, il retient une composante long terme, une composante cyclique et une composante court terme issues du rapport d'évidence variable par variable, puis construit un facteur composite principal à poids égaux. Il compare ensuite ce composite au score de la famille déjà présent dans le screen et teste marginalement chaque variable ajoutée au facteur existant.

Tous les résultats seront écrits dans exports/factor_family_pipeline_SMALL. Pour les périodes courtes, robust_score est utilisé à titre diagnostique et ne doit pas être comparé en niveau aux périodes historiques complètes.

In [ ]:

from pathlib import Path
import json
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "func.py").exists():
    raise RuntimeError(
        "Définissez le répertoire de travail Jupyter sur C:\\dev\\factor_backtest avant d'exécuter ce notebook."
    )

from func import (
    calculate_benchmark_performance,
    calculate_performance_ratios,
    combine_backtest_performances,
    export_backtest_results,
    load_backtest_data,
    test_composite_signals,
    test_incremental_signals,
)
from factor_config import signal_options

print("Les fonctions de recherche sont chargées.")


In [ ]:
MARKET = "EUROPE SMALL CAP"
BENCHMARK = "MSCI EUR SMALL"
START_DATE = "2007-12-01"
PERCENTILE = 0.13
N_JOBS = 1
PERIOD_BREAKPOINTS = [2009, 2013, 2017, 2020, 2022, 2024, 2026]
OUTPUT_NAME = "factor_family_pipeline_SMALL"
EVIDENCE_REPORT = Path(
    r"C:\dev\factor_backtest\exports\_agent_work\small_regime_fullpool.md"
)

BASELINE_CANDIDATES = {
    "growth": ("GROWTH_SCORE_FS_SECTOR", "Growth Avg Percentile"),
    "quality": ("Quality Avg Percentile", "MARGIN_SCORE_FS_SECTOR"),
    "momentum": ("MOMENTUM_SCORE_FS_SECTOR", "Mom Avg Percentile"),
    "value": ("VALUE_SCORE_FS_SECTOR", "Value Avg Percentile"),
    "dividend": ("Dividend Avg Percentile", "Dividend_NTM Avg Percentile"),
}
SELECTIONS = {
    "dividend": [
        {"role": "long", "variable": "PCT DvdYield FY1", "dimension": "level",
         "higher_is_better": True, "evidence_class": "long_core",
         "evidence_note": "Niveau de rendement FY1 stable sur plusieurs périodes"},
        {"role": "cycle", "variable": "DVD Yield FY0", "dimension": "rank_diff_6",
         "higher_is_better": True, "evidence_class": "cycle",
         "evidence_note": "Amélioration du rang du rendement sur une fenêtre longue"},
        {"role": "short", "variable": "DVD Yield FY1", "dimension": "pct_1",
         "higher_is_better": True, "evidence_class": "short_tactical",
         "evidence_note": "Variation courte du rendement FY1, utilisée comme composante tactique"},
    ],
    "growth": [
        {"role": "long", "variable": "PCT Hist GrossInc", "dimension": "rank_diff_3",
         "higher_is_better": True, "evidence_class": "long_core",
         "evidence_note": "Candidat inter-périodes fondé sur la tendance historique du gross income"},
        {"role": "cycle", "variable": "PCT Hist Sales", "dimension": "rank_diff_6",
         "higher_is_better": True, "evidence_class": "cycle",
         "evidence_note": "Composante lente de variation des ventes historiques"},
        {"role": "short", "variable": "Revenue 5Y CAGR", "dimension": "pct_1",
         "higher_is_better": True, "evidence_class": "short_tactical",
         "evidence_note": "Variation récente du chiffre d'affaires, forte mais peu couverte dans le rapport"},
    ],
    "momentum": [
        {"role": "long", "variable": "SP Price Target CIQ", "dimension": "pct_6",
         "higher_is_better": True, "evidence_class": "long_core",
         "evidence_note": "Variation moyen/long terme de l'objectif de cours"},
        {"role": "cycle", "variable": "SP Price Close CIQ", "dimension": "pct_12",
         "higher_is_better": True, "evidence_class": "cycle",
         "evidence_note": "Tendance de prix à horizon lent"},
        {"role": "short", "variable": "SP Price Target CIQ", "dimension": "pct_1",
         "higher_is_better": True, "evidence_class": "short_tactical",
         "evidence_note": "Révision courte de l'objectif de cours"},
    ],
    "quality": [
        {"role": "long", "variable": "PCT ROE", "dimension": "diff_3",
         "higher_is_better": True, "evidence_class": "long_core",
         "evidence_note": "Preuve inter-périodes stable d'amélioration du ROE"},
        {"role": "cycle", "variable": "Net Debt", "dimension": "diff_6",
         "higher_is_better": False, "evidence_class": "cycle",
         "evidence_note": "Baisse du levier ; direction lower-is-better explicitement définie"},
        {"role": "short", "variable": "Net Debt to Ebit", "dimension": "diff_1",
         "higher_is_better": False, "evidence_class": "short_tactical",
         "evidence_note": "Variation récente du désendettement, d'abord testée comme candidate incrémentale"},
    ],
    "value": [
        {"role": "long", "variable": "Earns Yield FY0", "dimension": "level",
         "higher_is_better": True, "evidence_class": "long_core",
         "evidence_note": "Niveau de rendement positif sur plusieurs périodes"},
        {"role": "cycle", "variable": "EV To EBITDA LTM", "dimension": "pct_3",
         "higher_is_better": False, "evidence_class": "cycle",
         "evidence_note": "Baisse du multiple de valorisation ; direction lower-is-better explicitement définie"},
        {"role": "short", "variable": "Earns Yield FY1", "dimension": "diff_1",
         "higher_is_better": True, "evidence_class": "short_tactical",
         "evidence_note": "Amélioration récente du rendement bénéficiaire forward"},
    ],
}


## 1. Sélection des composantes et règle de pondération égale

Chaque famille contient trois composantes : long terme, cycle et court terme ; chaque composante reçoit un poids de 1,0. Le score de la famille est obtenu par addition des contributions, puis transformé selon la neutralisation cross-sectionnelle déjà utilisée par le pipeline. Si une même variable brute apparaît deux fois avec des horizons différents, chaque dimension conserve sa contribution égale.

In [ ]:

def make_single_config(spec):
    kwargs = {"higher_is_better": bool(spec["higher_is_better"])}
    kwargs[spec["dimension"]] = 1.0
    return {spec["variable"]: signal_options(**kwargs)}


def make_family_config(specs):
    config = {}
    for spec in specs:
        variable = spec["variable"]
        if variable not in config:
            config[variable] = signal_options(
                higher_is_better=bool(spec["higher_is_better"])
            )
        config[variable][f"weight_{spec['dimension']}"] = 1.0
    return config


def make_baseline_config(family):
    variable = BASELINE_COLUMNS[family]
    return {variable: signal_options(level=1.0, higher_is_better=True)}


SELECTION_ROWS = []
for family, specs in SELECTIONS.items():
    for index, spec in enumerate(specs, start=1):
        SELECTION_ROWS.append(
            {
                "market": MARKET,
                "family": family,
                "component_index": index,
                "role": spec["role"],
                "variable": spec["variable"],
                "dimension": spec["dimension"],
                "higher_is_better": spec["higher_is_better"],
                "evidence_class": spec["evidence_class"],
                "evidence_note": spec["evidence_note"],
                "source_report": str(EVIDENCE_REPORT),
                "composite_weight": 1.0,
                "baseline_column": None,
            }
        )
SELECTION_MANIFEST = pd.DataFrame(SELECTION_ROWS)
display(SELECTION_MANIFEST)


In [ ]:

DATA_DIR = REPO_ROOT / "data"
SCREEN_PATH = DATA_DIR / "screen_aggregate.parquet"
RETURNS_PATH = DATA_DIR / "returns.parquet"
EXPORT_ROOT = REPO_ROOT / "exports"
EXPORT_DIR = EXPORT_ROOT / OUTPUT_NAME
LIST_NOIRE_PATH = None

try:
    import pyarrow.parquet as pq
    available_columns = set(pq.ParquetFile(SCREEN_PATH).schema_arrow.names)
except Exception as error:
    raise RuntimeError("Échec de lecture du schéma parquet du screen ; vérifiez que pyarrow est disponible.") from error

BASELINE_COLUMNS = {}
for family, candidates in BASELINE_CANDIDATES.items():
    selected = next((candidate for candidate in candidates if candidate in available_columns), None)
    if selected is None:
        raise KeyError(f"Aucune colonne de facteur existante pour {family} dans le screen : {candidates}")
    BASELINE_COLUMNS[family] = selected

if "SELECTION_MANIFEST" in globals():
    SELECTION_MANIFEST["baseline_column"] = SELECTION_MANIFEST["family"].map(BASELINE_COLUMNS)

SELECTED_RAW_VARIABLES = sorted(
    {
        spec["variable"]
        for specs in SELECTIONS.values()
        for spec in specs
    }
)
LOAD_VARIABLES = list(
    dict.fromkeys(SELECTED_RAW_VARIABLES + list(BASELINE_COLUMNS.values()))
)

screen, returns = load_backtest_data(
    screen_path=SCREEN_PATH,
    returns_path=RETURNS_PATH,
    variables=LOAD_VARIABLES,
    bench=BENCHMARK,
    start_date=START_DATE,
    lookback_periods=12,
    compact_dtypes=True,
)
screen["Date"] = pd.to_datetime(screen["Date"])

missing = [column for column in LOAD_VARIABLES if column not in screen.columns]
if missing:
    raise KeyError(f"Variables absentes après chargement : {missing}")
if f"Weight in {BENCHMARK}" not in screen.columns:
    raise KeyError(f"La colonne Weight in {BENCHMARK} est absente du screen")

BENCH_PERF = calculate_benchmark_performance(
    screen=screen,
    returns=returns,
    bench=BENCHMARK,
    start_date=START_DATE,
)

MONTHLY_BASE_CACHE = {}
RUN_OPTIONS = {
    "bench": BENCHMARK,
    "bench_perf": BENCH_PERF,
    "percentile": PERCENTILE,
    "start_date": START_DATE,
    "freq_rebal": 1,
    "fill_method": "copy",
    "n_jobs": N_JOBS,
    "retain_builders": False,
    "monthly_base_cache": MONTHLY_BASE_CACHE,
    "period_breakpoints": PERIOD_BREAKPOINTS,
    "show_plot": False,
    "build_figure": False,
}

print(f"screen={screen.shape}, returns={returns.shape}")
print(f"benchmark={BENCHMARK}; baseline columns={BASELINE_COLUMNS}")


## 2. Construire et backtester les nouveaux composites familiaux et les facteurs existants du screen

In [ ]:

COMPOSITE_CONFIGS = {}
for family, specs in SELECTIONS.items():
    COMPOSITE_CONFIGS[f"new_family_{family}"] = make_family_config(specs)
for family in SELECTIONS:
    COMPOSITE_CONFIGS[f"screen_baseline_{family}"] = make_baseline_config(family)

composite_batch = test_composite_signals(
    screen=screen,
    returns=returns,
    composite_configs=COMPOSITE_CONFIGS,
    list_noire_path=LIST_NOIRE_PATH,
    score_prefix="Score_FamilyPipeline",
    **RUN_OPTIONS,
)
screen = composite_batch["screen"]
print("Le backtest comparatif des nouveaux composites familiaux et des facteurs existants du screen est terminé.")


## 3. Analyse incrémentale de chaque variable

Chaque batch repart du même facteur de référence de la famille et ajoute une seule composante variable × dimension. Les colonnes delta_active_cagr, delta_top_worst_cagr, delta_top_information_ratio, delta_robust_score, delta_active_max_drawdown et delta_tracking_error_annualized permettent de mesurer directement la contribution marginale.

In [ ]:

incremental_batches = {}
for family, specs in SELECTIONS.items():
    for index, spec in enumerate(specs, start=1):
        batch_key = f"{family}__{spec['role']}__{index}"
        incremental_batches[batch_key] = test_incremental_signals(
            screen=screen,
            returns=returns,
            baseline_config=make_baseline_config(family),
            candidate_config=make_single_config(spec),
            list_noire_path=LIST_NOIRE_PATH,
            **RUN_OPTIONS,
        )
        screen = incremental_batches[batch_key]["screen"]

all_results = {
    "composite_comparison": composite_batch,
    "incremental": incremental_batches,
}
print(f"{len(incremental_batches)} batches incrémentaux à une composante sont terminés.")


## 4. Export normalisé

L'export conserve les métriques officielles, les courbes de performance, le tableau de comparaison composite/facteur existant, le tableau incremental par composante et le manifest de sélection. Il ne faut pas regarder uniquement le CAGR total : la sélection doit combiner les périodes économiques complètes, Top/Worst, IR, drawdown, tracking error et les gates.

In [ ]:

exported = export_backtest_results(
    results=all_results,
    output_dir=EXPORT_ROOT,
    export_name=OUTPUT_NAME,
    export_html=False,
    export_png=False,
    export_holdings=False,
)
EXPORT_DIR = Path(exported["export_dir"])

metrics = pd.read_csv(EXPORT_DIR / "backtest_metrics.csv")
with (EXPORT_DIR / "backtest_registry.json").open("r", encoding="utf-8") as handle:
    registry = json.load(handle)
path_by_name = {
    entry.get("metadata", {}).get("test_name"): entry.get("test_path")
    for entry in registry
    if entry.get("metadata", {}).get("test_name") and entry.get("test_path")
}

METRIC_COLUMNS = [
    "active_cagr",
    "top_worst_cagr",
    "top_information_ratio",
    "robust_score",
    "active_max_drawdown",
    "tracking_error_annualized",
    "min_rolling_3y_cagr",
    "top_bench_ratio",
    "top_worst_ratio",
    "top_annualized_return",
    "bench_annualized_return",
    "observation_count",
    "years",
]
COMPARABILITY_COLUMNS = [
    "robust_score_comparable"
] if "robust_score_comparable" in metrics.columns else []


def _path_for(test_name):
    if test_name not in path_by_name:
        raise KeyError(f"Le test_name={test_name} est absent du registry exporté")
    return path_by_name[test_name]


def _metric_slice(test_path):
    return metrics.loc[metrics["test_path"].eq(test_path)].copy()


family_comparison_parts = []
family_names = list(SELECTIONS)
for family in family_names:
    new_rows = _metric_slice(_path_for(f"new_family_{family}"))
    base_rows = _metric_slice(_path_for(f"screen_baseline_{family}"))
    left_columns = ["period_id", "scope", "period_label", *METRIC_COLUMNS, *COMPARABILITY_COLUMNS]
    right_columns = ["period_id", "scope", *METRIC_COLUMNS, *COMPARABILITY_COLUMNS]
    left = new_rows[left_columns].rename(
        columns={column: f"{column}_new" for column in [*METRIC_COLUMNS, *COMPARABILITY_COLUMNS]}
    )
    right = base_rows[right_columns].rename(
        columns={column: f"{column}_screen" for column in [*METRIC_COLUMNS, *COMPARABILITY_COLUMNS]}
    )
    joined = left.merge(right, on=["period_id", "scope"], how="outer")
    joined.insert(0, "family", family)
    for column in METRIC_COLUMNS:
        joined[f"delta_{column}"] = (
            joined[f"{column}_new"] - joined[f"{column}_screen"]
        )
    joined["new_perf_gate"] = (
        joined["active_cagr_new"].gt(0)
        & joined["top_worst_cagr_new"].gt(0)
        & joined["top_information_ratio_new"].gt(0)
    )
    joined["screen_perf_gate"] = (
        joined["active_cagr_screen"].gt(0)
        & joined["top_worst_cagr_screen"].gt(0)
        & joined["top_information_ratio_screen"].gt(0)
    )
    joined["performance_improved"] = (
        joined["delta_active_cagr"].gt(0)
        & joined["delta_top_worst_cagr"].gt(0)
        & joined["delta_top_information_ratio"].gt(0)
    )
    joined["risk_not_worse"] = (
        joined["delta_active_max_drawdown"].le(0)
        & joined["delta_tracking_error_annualized"].le(0)
    )
    if COMPARABILITY_COLUMNS:
        comparable_period = (
            joined["scope"].eq("total")
            | (
                joined["robust_score_comparable_new"].astype(str).str.lower().isin(["true", "1", "yes"])
                & joined["robust_score_comparable_screen"].astype(str).str.lower().isin(["true", "1", "yes"])
            )
        )
    else:
        comparable_period = joined["scope"].eq("total")
    joined["strict_comparable_improvement"] = (
        comparable_period
        & joined["performance_improved"]
        & joined["risk_not_worse"]
        & joined["robust_score_new"].gt(joined["robust_score_screen"])
    )
    family_comparison_parts.append(joined)

family_comparison = pd.concat(family_comparison_parts, ignore_index=True)
family_comparison.to_csv(
    EXPORT_DIR / "family_composite_vs_screen.csv", index=False
)
family_comparison.loc[family_comparison["period_id"].eq("total")].to_csv(
    EXPORT_DIR / "family_composite_vs_screen_total.csv", index=False
)

incremental_lookup = {}
for family, specs in SELECTIONS.items():
    for index, spec in enumerate(specs, start=1):
        incremental_lookup[f"{family}__{spec['role']}__{index}"] = {
            "family": family,
            "role": spec["role"],
            "variable": spec["variable"],
            "dimension": spec["dimension"],
            "higher_is_better": spec["higher_is_better"],
        }

incremental_rows = []
for test_group in metrics.loc[
    metrics["test_type"].isin(
        ["incremental_baseline", "incremental_candidate"]
    ),
    "test_group",
].dropna().unique():
    group_rows = metrics.loc[metrics["test_group"].eq(test_group)].copy()
    baseline_rows = group_rows.loc[
        group_rows["test_type"].eq("incremental_baseline")
    ]
    candidate_rows = group_rows.loc[
        group_rows["test_type"].eq("incremental_candidate")
    ]
    batch_key = str(test_group).split(" / ")[-1]
    spec_info = incremental_lookup.get(batch_key, {})
    for _, candidate in candidate_rows.iterrows():
        baseline = baseline_rows.loc[
            baseline_rows["period_id"].eq(candidate["period_id"])
        ]
        if baseline.empty:
            continue
        baseline = baseline.iloc[0]
        row = {
            "batch_key": batch_key,
            **spec_info,
            "period_id": candidate["period_id"],
            "scope": candidate["scope"],
            "period_label": candidate.get("period_label"),
            "candidate_test_path": candidate["test_path"],
            "baseline_test_path": baseline["test_path"],
        }
        for column in METRIC_COLUMNS:
            row[f"{column}_candidate"] = candidate.get(column)
            row[f"{column}_baseline"] = baseline.get(column)
            row[f"delta_{column}"] = (
                candidate.get(column) - baseline.get(column)
            )
        row["candidate_perf_gate"] = (
            candidate["active_cagr"] > 0
            and candidate["top_worst_cagr"] > 0
            and candidate["top_information_ratio"] > 0
        )
        row["baseline_perf_gate"] = (
            baseline["active_cagr"] > 0
            and baseline["top_worst_cagr"] > 0
            and baseline["top_information_ratio"] > 0
        )
        row["incremental_perf_improved"] = (
            row["delta_active_cagr"] > 0
            and row["delta_top_worst_cagr"] > 0
            and row["delta_top_information_ratio"] > 0
        )
        row["incremental_risk_not_worse"] = (
            row["delta_active_max_drawdown"] <= 0
            and row["delta_tracking_error_annualized"] <= 0
        )
        incremental_rows.append(row)

incremental_effects = pd.DataFrame(incremental_rows)
incremental_effects.to_csv(EXPORT_DIR / "incremental_effects.csv", index=False)
incremental_effects.loc[
    incremental_effects["period_id"].eq("total")
].to_csv(EXPORT_DIR / "incremental_effects_total.csv", index=False)

performance_selections = {}
first_baseline_path = None
for family in family_names:
    new_path = _path_for(f"new_family_{family}")
    base_path = _path_for(f"screen_baseline_{family}")
    performance_selections[f"{family}_new"] = (new_path, "Top")
    performance_selections[f"{family}_screen"] = (base_path, "Top")
    first_baseline_path = first_baseline_path or base_path
performance_selections["Benchmark"] = (first_baseline_path, "Bench")

top_curves = combine_backtest_performances(
    export_dir=EXPORT_DIR,
    selections=performance_selections,
)
top_curves.to_csv(EXPORT_DIR / "performance_top_curves.csv", index=True)
top_ratios = calculate_performance_ratios(top_curves, benchmark_column="Benchmark")
top_ratios.to_csv(EXPORT_DIR / "performance_ratios.csv", index=True)

run_manifest = {
    "market": MARKET,
    "benchmark": BENCHMARK,
    "start_date": START_DATE,
    "period_breakpoints": PERIOD_BREAKPOINTS,
    "percentile": PERCENTILE,
    "rebalancing_frequency": 1,
    "fill_method": "copy",
    "baseline_columns": BASELINE_COLUMNS,
    "selection_manifest": SELECTION_ROWS,
    "source_evidence_report": str(EVIDENCE_REPORT),
    "gate": {
        "performance": "active_cagr > 0 and top_worst_cagr > 0 and top_information_ratio > 0",
        "strict_comparable": "performance gate and robust_score > 0",
        "short_period_note": "robust_score is diagnostic when robust_score_comparable is false",
    },
    "outputs": [
        "backtest_metrics.csv",
        "backtest_registry.json",
        "family_composite_vs_screen.csv",
        "family_composite_vs_screen_total.csv",
        "incremental_effects.csv",
        "incremental_effects_total.csv",
        "performance_top_curves.csv",
        "performance_ratios.csv",
        "selection_manifest.csv",
        "run_manifest.json",
    ],
}
SELECTION_MANIFEST.to_csv(EXPORT_DIR / "selection_manifest.csv", index=False)
with (EXPORT_DIR / "run_manifest.json").open("w", encoding="utf-8") as handle:
    json.dump(run_manifest, handle, ensure_ascii=False, indent=2)

print(f"Répertoire des résultats : {EXPORT_DIR}")
display(
    family_comparison.loc[
        family_comparison["period_id"].eq("total"),
        [
            "family",
            "active_cagr_new",
            "active_cagr_screen",
            "delta_active_cagr",
            "top_information_ratio_new",
            "top_information_ratio_screen",
            "delta_top_information_ratio",
            "robust_score_new",
            "robust_score_screen",
            "strict_comparable_improvement",
        ],
    ].sort_values("delta_active_cagr", ascending=False)
)
display(
    incremental_effects.loc[
        incremental_effects["period_id"].eq("total"),
        [
            "family",
            "role",
            "variable",
            "dimension",
            "delta_active_cagr",
            "delta_top_worst_cagr",
            "delta_top_information_ratio",
            "delta_robust_score",
            "incremental_perf_improved",
            "incremental_risk_not_worse",
        ],
    ].sort_values(
        ["family", "delta_active_cagr"], ascending=[True, False]
    )
)


## 5. Ordre de lecture après exécution

1. Consulter family_composite_vs_screen_total.csv pour vérifier la direction de l'écart total entre le nouveau composite et le facteur de référence du screen.
2. Consulter les lignes par période de family_composite_vs_screen.csv pour vérifier que l'amélioration ne provient pas d'une seule période ou d'un échantillon court.
3. Consulter incremental_effects_total.csv et incremental_effects.csv : une composante ne doit être proposée à l'ajout que si les métriques de performance s'améliorent sans détérioration du risque.
4. Revenir ensuite à backtest_metrics.csv et backtest_registry.json afin de vérifier la composition, les périodes, le benchmark, observation_count et la provenance.

Ce notebook utilise le sous-ensemble Top12 fourni par le rapport d'évidence ; les résultats ne doivent donc pas être interprétés comme une sélection non biaisée de l'univers complet des variables.